# 🌒 EX0005Aa | The Examiner Must Discover

**New experiment, NOT a rerun or rescore of EX0005A.** Original November 15 work ended at 20:00 America/Denver. This notebook will **not** make a billable API call until the user freezes a **later** distinct session and explicitly opts in.

Two distinct measurements: (1) matched-output counterfactual signs with identical tool calls, (2) independent fault discovery on a new developmental fixture compared with a heuristic. Neither proves general mechanistic understanding or blinded efficacy.

Work one cell at a time. Save errors. Never paste your API secret into a cell. No data from sealed patients.

## Cell 1 · Install local dependencies
No OpenAI requests. A CPU notebook is sufficient.

In [ ]:
%pip -q install "openai>=2.0" numpy torch

## Cell 2 · Reconstruct the exact experimental scripts
The scripts are embedded in this notebook, so no GitHub checkout or upload of a ZIP is needed. This cell contains source code, not API generation.

In [ ]:
from pathlib import Path
Path('ex0005aa_public.py').write_text('"""EX0005Aa fresh public developmental fixtures; independent numpy + torch checks.\n\nNo private/healthy weights, planted indices or evaluator-only data go to tool responses.\nNo sealed patients or data. Predictions are checked on separate synthetic development examples.\n"""\nfrom __future__ import annotations\nimport copy, hashlib, json\nfrom dataclasses import dataclass\nfrom typing import Any\nimport numpy as np\nimport torch\n\nCASES = (\'C_positive\',\'C_negative\',\'D_unknown\')\nEVAL_SEEDS = {\'C_positive\':150619,\'C_negative\':150619,\'D_unknown\':150631}\nPROBE_PACKS = {\n  \'feature0\': ((0.5,0.0),(1.0,0.0)),\n  \'feature1\': ((0.0,0.5),(0.0,1.0)),\n  \'mixed\': ((0.8,0.9),(1.25,0.6)),\n}\nTENSOR_SIZES = {\'W1\':4,\'b1\':2,\'W2\':2,\'b2\':1}\n\n\ndef dumpable(x: Any)->Any:\n    if isinstance(x,dict):return {str(k):dumpable(v) for k,v in x.items()}\n    if isinstance(x,(list,tuple)):return [dumpable(v) for v in x]\n    if isinstance(x,np.ndarray):return x.tolist()\n    if isinstance(x,np.generic):return x.item()\n    return x\n\n\ndef fingerprint(x:Any)->str:\n    return hashlib.sha256(json.dumps(dumpable(x),sort_keys=True,separators=(\',\',\':\')).encode()).hexdigest()\n\n\ndef healthy_parameters()->dict[str,np.ndarray]:\n    return {\'W1\':np.eye(2,dtype=float),\'b1\':np.zeros(2,dtype=float),\n            \'W2\':np.array([1.0,0.5],dtype=float),\'b2\':np.array([0.1],dtype=float)}\n\n\ndef predict_np(w, x):\n    return np.maximum(np.asarray(x,dtype=float) @ w[\'W1\'].T + w[\'b1\'],0) @ w[\'W2\'] + w[\'b2\'][0]\n\n\ndef predict_torch(w,x):\n    with torch.no_grad():\n        a=torch.tensor(np.asarray(x,dtype=float),dtype=torch.float64)\n        h=torch.relu(a @ torch.tensor(w[\'W1\'],dtype=torch.float64).T + torch.tensor(w[\'b1\'],dtype=torch.float64))\n        return (h @ torch.tensor(w[\'W2\'],dtype=torch.float64) + torch.tensor(w[\'b2\'][0],dtype=torch.float64)).numpy()\n\n\ndef mse(w,x,y):return float(np.mean((predict_np(w,x)-y)**2))\n\n@dataclass\nclass Patient:\n    case:str\n    w:dict\n    healthy:dict\n    x_eval:np.ndarray\n    y_eval:np.ndarray\n\n\ndef make_case(case):\n    if case not in CASES:raise ValueError(\'unknown case\')\n    h=healthy_parameters();w=copy.deepcopy(h)\n    if case==\'C_positive\':w[\'W2\'][1]+=0.75\n    if case==\'C_negative\':w[\'W2\'][1]-=0.75\n    if case==\'D_unknown\':w[\'W1\'][0,0]-=1.30\n    x=np.random.default_rng(EVAL_SEEDS[case]).uniform(.2,1.5,size=(80,2))\n    return Patient(case,w,h,x,predict_np(h,x))\n\n\ndef inspect_output_weights(p):\n    return {\'tensor\':\'W2\',\'shape\':[2],\'values\':p.w[\'W2\'].tolist(),\'provenance\':\'corrupted_checkpoint\'}\n\n\ndef probe_feature1(p):\n    return probe_pack(p,\'feature1\')\n\n\ndef inspect_tensor(p,tensor):\n    if tensor not in TENSOR_SIZES:raise ValueError(\'bad tensor\')\n    return {\'tensor\':tensor,\'shape\':list(p.w[tensor].shape),\'values\':p.w[tensor].ravel().tolist(), \'provenance\':\'corrupted_checkpoint\'}\n\n\ndef probe_pack(p,pack):\n    if pack not in PROBE_PACKS:raise ValueError(\'bad pack\')\n    x=np.asarray(PROBE_PACKS[pack],dtype=float)\n    actual=predict_np(p.w,x);expected=predict_np(p.healthy,x)\n    return {\'pack\':pack,\'rows\':[{\'x\':xx.tolist(),\'target\':float(yy),\'prediction\':float(aa),\'residual\':float(aa-yy)} for xx,yy,aa in zip(x,expected,actual)],\'provenance\':\'public_development_examples\'}\n\n\ndef apply_edit(p,edits):\n    if not isinstance(edits,list) or not 1<=len(edits)<=4:raise ValueError(\'edit count\')\n    w=copy.deepcopy(p.w);seen=set()\n    for e in edits:\n        if not isinstance(e,dict) or set(e)!={\'tensor\',\'index\',\'delta\'}:raise ValueError(\'edit schema\')\n        t,i,d=e[\'tensor\'],e[\'index\'],e[\'delta\']\n        if t not in w or type(i) is not int or not 0<=i<w[t].size or type(d) not in (int,float) or not np.isfinite(d) or abs(d)>2.0 or (t,i) in seen:raise ValueError(\'invalid edit\')\n        seen.add((t,i));w[t].flat[i]+=d\n    return w\n\n\ndef score_edits(p,edits):\n    altered=apply_edit(p,edits)\n    return {\'before_mse\':mse(p.w,p.x_eval,p.y_eval),\'after_mse\':mse(altered,p.x_eval,p.y_eval), \'edits\':edits}\n\n\ndef oracle(p):\n    out=[]\n    for t in TENSOR_SIZES:\n        for i,(a,b) in enumerate(zip(p.w[t].ravel(),p.healthy[t].ravel())):\n            if abs(a-b)>1e-12:out.append({\'tensor\':t,\'index\':i,\'delta\':float(b-a)})\n    return out\n\n\ndef simple_heuristic(p):\n    # Fully explicit baseline using *public* probe residual and current architecture;\n    # corrects an output-layer weight only. A zero residual produces no patch.\n    r=probe_pack(p,\'feature1\')[\'rows\'][1]\n    delta=-r[\'residual\']/r[\'x\'][1]\n    if abs(delta)<1e-12:return None\n    return [{\'tensor\':\'W2\',\'index\':1,\'delta\':float(delta)}]\n\n\ndef intervention_effect(p):\n    after=apply_edit(p,[{\'tensor\':\'W2\',\'index\':1,\'delta\':-0.25}])\n    return mse(after,p.x_eval,p.y_eval)-mse(p.w,p.x_eval,p.y_eval)\n\n\ndef qualification():\n    p={name:make_case(name) for name in CASES}\n    assert np.array_equal(p[\'C_positive\'].x_eval,p[\'C_negative\'].x_eval)\n    assert np.array_equal(p[\'C_positive\'].y_eval,p[\'C_negative\'].y_eval)\n    for t in TENSOR_SIZES:\n        if t!=\'W2\':assert np.array_equal(p[\'C_positive\'].w[t],p[\'C_negative\'].w[t])\n    assert p[\'C_positive\'].w[\'W2\'][0] == p[\'C_negative\'].w[\'W2\'][0]\n    assert intervention_effect(p[\'C_positive\'])<0<intervention_effect(p[\'C_negative\'])\n    result={}\n    for case,patient in p.items():\n        assert np.allclose(predict_np(patient.w,patient.x_eval),predict_torch(patient.w,patient.x_eval),atol=1e-12)\n        base=mse(patient.w,patient.x_eval,patient.y_eval)\n        oracle_result=score_edits(patient,oracle(patient))\n        zeros=[]\n        # Naive reference: zero the first output weight (doesn\'t use lesion key)\n        zeros=[{\'tensor\':\'W2\',\'index\':0,\'delta\':-float(patient.w[\'W2\'][0])}]\n        zero_score=score_edits(patient,zeros)\n        heu=simple_heuristic(patient)\n        heur_score=score_edits(patient,heu) if heu else None\n        assert base>1e-4 and oracle_result[\'after_mse\'] < 1e-20\n        result[case]={\'corrupt_mse\':base,\'oracle_mse\':oracle_result[\'after_mse\'],\'zero_mse\':zero_score[\'after_mse\'],\n                      \'heuristic_mse\':heur_score[\'after_mse\'] if heur_score else base,\n                      \'counterfactual_delta_mse\':intervention_effect(patient)}\n    for fn in [lambda:inspect_tensor(p[\'D_unknown\'],\'garbage\'),lambda:probe_pack(p[\'D_unknown\'],\'invalid\'),\n               lambda:apply_edit(p[\'C_positive\'],[{\'tensor\':\'W2\',\'index\':4,\'delta\':0.2}])]:\n        try:fn();raise AssertionError(\'bad request accepted\')\n        except ValueError:pass\n    return {\'status\':\'G0_LOCAL_PASS_ONLY\',\'cases\':result,\'fixture_sha256\':fingerprint({name:{t:arr.tolist() for t,arr in patient.w.items()} for name,patient in p.items()}),\n            \'precommitted_paired_direction\':\'C_positive negative, C_negative positive\',\n            \'seeds\':EVAL_SEEDS,\'invalid_controls_rejected\':3}\n\nif __name__==\'__main__\': print(json.dumps(qualification(),indent=2))\n', encoding='utf-8')
print('Wrote', 'ex0005aa_public.py', Path('ex0005aa_public.py').stat().st_size, 'bytes')

In [ ]:
from pathlib import Path
Path('run_luna_aa.py').write_text('"""EX0005Aa new-session, public development-only GPT-6 Luna investigator.\n\nNo calls on import. No duplicate live launches. Separate protocol freeze & model contact.\nUses Responses API native tools, exactly one tool call per observation response.\nResults persist after EVERY response. API error -> local evidence preserved, no retry.\n"""\nfrom __future__ import annotations\nimport argparse, datetime as dt, hashlib, importlib.metadata, json, os, pathlib, platform, sys\nfrom zoneinfo import ZoneInfo\nimport numpy as np\nfrom ex0005aa_public import (make_case, qualification, inspect_output_weights, probe_feature1,\n    inspect_tensor, probe_pack, score_edits, oracle, simple_heuristic, intervention_effect,\n    TENSOR_SIZES, fingerprint, dumpable)\n\nMODEL=\'gpt-6-luna\'\nREASONING={\'effort\':\'low\'}\nOUTPUT_CAP=1800\nMAX_RESPONSES=14\nMAX_INPUT_TOKENS=80000\nMAX_OUTPUT_TOKENS=24000\nDIAG_STEPS_DISCOVERY=4\nOUTDIR=pathlib.Path(\'ex0005aa_receipts\')\nSOURCE_FILES=[\'ex0005aa_public.py\',\'run_luna_aa.py\']\nHISTORICAL_SHA={\n \'EX0005A_ACTION_JOURNAL.json\':\'48d0f918dfc1c0b42a935a3af8f12d2189ed0318a6b72dfdcedd20b1d44ddb72\',\n \'EX0005A_RESULTS.json\':\'16bd3d9dd5c9346a06a2b3ea9d2e432f86d42b70264a6cbe4687cd5d85e6178c\'\n}\nPAIR_TOOLS=[\n {\'type\':\'function\',\'name\':\'inspect_output_weights\',\n  \'description\':\'Read the two scalar weights of the corrupted network output layer. This returns no healthy or oracle weights.\',\n  \'strict\':True,\'parameters\':{\'type\':\'object\',\'properties\':{},\'required\':[],\'additionalProperties\':False}},\n {\'type\':\'function\',\'name\':\'probe_feature1\',\n  \'description\':\'Observe two FIXED public development rows, feature1 inputs x=[0,0.5] and [0,1.0], with predictions and target residuals. No indices to guess.\',\n  \'strict\':True,\'parameters\':{\'type\':\'object\',\'properties\':{},\'required\':[],\'additionalProperties\':False}}\n]\nDISCOVERY_TOOLS=[\n {\'type\':\'function\',\'name\':\'inspect_weights\',\n  \'description\':\'Read ONE complete current corrupted tensor, with no healthy values or fault information. W1 has 4 flat elements, b1 has 2, W2 has 2, b2 has 1.\',\n  \'strict\':True,\'parameters\':{\'type\':\'object\',\'properties\':{\'tensor\':{\'type\':\'string\',\'enum\':[\'W1\',\'b1\',\'W2\',\'b2\']}},\n                             \'required\':[\'tensor\'],\'additionalProperties\':False}},\n {\'type\':\'function\',\'name\':\'probe_examples\',\n  \'description\':\'Read an entire small, fixed public probe pack with predictions, targets and residuals. Valid packs: feature0, feature1, mixed. No indices.\',\n  \'strict\':True,\'parameters\':{\'type\':\'object\',\'properties\':{\'pack\':{\'type\':\'string\',\'enum\':[\'feature0\',\'feature1\',\'mixed\']}},\n                             \'required\':[\'pack\'],\'additionalProperties\':False}}\n]\nPAIR_BASE=(\'You are studying a synthetic 2-input, 2-ReLU-hidden, 1-linear-output network. \'\n           \'No access to the healthy checkpoint. Use only the numeric observations you are given. \'\n           \'This question measures an explicit counterfactual prediction, NOT discovery of a hidden fault. \'\n           \'After observing both tools, predict the change in mean squared error on a separate public development set \'\n           \'if W2 flat index 1 were decreased by exactly 0.25. Do not invent unobserved activations or claim mechanistic proof.\')\nPAIR_COMMIT=(\'Using ONLY observed numerical evidence, commit one JSON object with EXACT keys: \'\n             \'predicted_delta_mse (finite number; after minus before), \'\n             \'uncertainty (low/medium/high), explanation (under 70 words). \'\n             \'No code fences, and no additional tools.\')\nDISCOVERY_BASE=(\'You are examining a public synthetic network with inputs x0,x1, 2 ReLU hidden units, \'\n                \'one linear output and exactly one planted scalar parameter corruption of unknown location and sign. \'\n                \'Tensor names are W1 (shape 2x2; row-major flattened), b1 (2), W2 (2), b2 (1). \'\n                \'Choose your own bounded numerical diagnostic calls. No healthy weights, oracle or answer keys are available. \'\n                \'A simple output-weight heuristic will be scored as a baseline. \'\n                \'Do not assume the apparent largest weight is the lesion; use input-output evidence.\')\nDISCOVERY_COMMIT=(\'Commit now; no further tools. Return EXACT JSON keys: fault_region (format W1[0], b1[1] etc), \'\n                  \'uncertainty (low/medium/high), patch (array of zero to four {tensor,index,delta} scalar edits), \'\n                  \'explanation (max 70 words). Empty patch explicitly abstains. No additional keys or code fences.\')\n\n\ndef sha(path):return hashlib.sha256(pathlib.Path(path).read_bytes()).hexdigest()\n\ndef atomic_json(path,obj):\n    path=pathlib.Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    tmp=path.with_name(path.name+\'.tmp\')\n    tmp.write_text(json.dumps(dumpable(obj),indent=2,sort_keys=True,allow_nan=False))\n    tmp.replace(path)\n\ndef load_json(path):return json.loads(pathlib.Path(path).read_text())\n\ndef human_fields(filename):\n    data=load_json(filename)\n    keys=(\'human_predicted_failure\',\'human_predicted_contrast\',\'human_false_positive_test\',\'human_unsupported_mechanism\')\n    for key in keys:\n        s=data.get(key)\n        if not isinstance(s,str) or len(s.strip())<20 or any(x in s.upper() for x in [\'TO FILL\',\'PLACEHOLDER\']):\n            raise ValueError(\'missing human prediction: \'+key)\n    if data.get(\'assistance_disclosure\') not in (\'I\',\'H\',\'C\',\'D\'):raise ValueError(\'disclose assistance\')\n    if data.get(\'origin\')!=\'carried_from_EX0005A_pre_live\':raise ValueError(\'mark prior predictions as carried from EX0005A\')\n    if data.get(\'approved_as_prior_for_aa\') is not True:raise ValueError(\'human must approve using prior predictions\')\n    return data\n\ndef code_hashes():\n    folder=pathlib.Path(__file__).resolve().parent\n    return {name:sha(folder/name) for name in SOURCE_FILES}\n\ndef local_date_time():return dt.datetime.now(ZoneInfo(\'America/Denver\'))\n\ndef time_window(manifest,now=None):\n    now=now or local_date_time()\n    day=dt.date.fromisoformat(manifest[\'session_date\'])\n    if day<=dt.date(2026,11,15):raise RuntimeError(\'EX0005Aa may not extend the closed EX0005A session\')\n    start_h,start_m=map(int,manifest[\'session_start\'].split(\':\'))\n    end_h,end_m=map(int,manifest[\'session_end\'].split(\':\'))\n    start=dt.datetime.combine(day,dt.time(start_h,start_m),tzinfo=ZoneInfo(\'America/Denver\'))\n    end=dt.datetime.combine(day,dt.time(end_h,end_m),tzinfo=ZoneInfo(\'America/Denver\'))\n    if not (0<(end-start).total_seconds()<=4*3600):raise RuntimeError(\'invalid session duration\')\n    if not (start<=now<end):raise RuntimeError(\'outside prospectively frozen session window\')\n\n\ndef freeze(cfg,session_date,start,end):\n    OUTDIR.mkdir(exist_ok=True)\n    filename=OUTDIR/\'PRE_LIVE_FREEZE.json\'\n    if filename.exists() or (OUTDIR/\'LIVE_ATTEMPT_STARTED.txt\').exists():\n        raise RuntimeError(\'freeze already exists; do not replace it\')\n    day=dt.date.fromisoformat(session_date)\n    if day<=dt.date(2026,11,15):raise RuntimeError(\'must be distinct session after EX0005A\')\n    for s in (start,end):dt.time.fromisoformat(s)\n    start_dt=dt.datetime.combine(day,dt.time.fromisoformat(start))\n    end_dt=dt.datetime.combine(day,dt.time.fromisoformat(end))\n    if not (dt.timedelta(0)<end_dt-start_dt<=dt.timedelta(hours=4)):\n        raise RuntimeError(\'separate session must be 4h or less\')\n    q=qualification()\n    manifest={\'experiment\':\'EX0005Aa\',\'frozen_at\':local_date_time().isoformat(),\n      \'session_date\':session_date,\'session_start\':start,\'session_end\':end,\n      \'model\':MODEL,\'reasoning\':REASONING,\'max_output_tokens\':OUTPUT_CAP,\n      \'max_response_calls\':MAX_RESPONSES,\'input_token_cap\':MAX_INPUT_TOKENS,\'output_token_cap\':MAX_OUTPUT_TOKENS,\n      \'discovery_diagnostic_steps\':DIAG_STEPS_DISCOVERY,\'paired_protocol\':\'fixed identical 2-tool sequence\',\n      \'discovery_protocol\':\'free-choice 4 bounded calls; separate from paired contrast\',\n      \'public_fixture_fingerprint\':q[\'fixture_sha256\'],\n      \'source_sha256\':code_hashes(), \'prior_artifact_sha256\':HISTORICAL_SHA,\n      \'human_fields\':cfg,\'no_sealed_test\':True,\n      \'metrics_and_gates\':{\n       \'G0\':\'analytical oracle restores cases; heuristic/noedit controls and numpy/torch agreement\',\n       \'G1\':\'both paired cases yield exactly two native valid forced calls; report discovery use separately\',\n       \'G2\':\'paired cases have opposing, oracle-correct signed prediction under exact matching calls\',\n       \'G3\':\'both paired cases produce schema-valid commitments; abstentions reported separately\',\n       \'G4\':\'secondary discovery: model patch has lower MSE than no-edit AND simple heuristic by >=0.05 absolute, and >=50% recovery; otherwise FAIL or NOT RUN\',\n       \'mechanism\':\'explanation only bounded and not validated by mere patch success\'\n      },\n      \'software\':{\'python\':platform.python_version(),\'numpy\':np.__version__,\n                  \'torch\':importlib.metadata.version(\'torch\'),\n                  \'openai\':importlib.metadata.version(\'openai\') if importlib.util.find_spec(\'openai\') else \'not installed\'}}\n    manifest[\'freeze_sha256\']=fingerprint(manifest)\n    atomic_json(filename,manifest)\n    return manifest\n\n\ndef api_key():\n    token=os.environ.get(\'OPENAI_API_KEY\',\'\')\n    if not token:\n        try:\n            from google.colab import userdata\n            token=userdata.get(\'OPENAI_API_KEY\') or \'\'\n        except ImportError:pass\n    if not token:raise RuntimeError(\'API key missing; no request\')\n    return token\n\n\ndef reject_secret(msg):\n    # Log no raw exception bodies or authorization headers.\n    return type(msg).__name__\n\n\ndef validate_pair_commit(obj):\n    if not isinstance(obj,dict) or set(obj)!={\'predicted_delta_mse\',\'uncertainty\',\'explanation\'}:raise ValueError(\'pair commitment keys\')\n    v=obj[\'predicted_delta_mse\']\n    if type(v) not in (float,int) or not np.isfinite(v):raise ValueError(\'prediction nonnumeric\')\n    if obj[\'uncertainty\'] not in (\'low\',\'medium\',\'high\'):raise ValueError(\'uncertainty\')\n    if not isinstance(obj[\'explanation\'],str) or not 0<len(obj[\'explanation\'].split())<=70:raise ValueError(\'explanation\')\n    return obj\n\n\ndef validate_discovery_commit(obj):\n    if not isinstance(obj,dict) or set(obj)!={\'fault_region\',\'uncertainty\',\'patch\',\'explanation\'}:raise ValueError(\'discovery commitment keys\')\n    import re\n    m=re.fullmatch(r\'(W1|b1|W2|b2)\\[(\\d+)\\]\',obj[\'fault_region\']) if isinstance(obj[\'fault_region\'],str) else None\n    if not m or int(m.group(2))>=TENSOR_SIZES[m.group(1)]:raise ValueError(\'invalid fault region\')\n    if obj[\'uncertainty\'] not in (\'low\',\'medium\',\'high\'):raise ValueError(\'uncertainty\')\n    if not isinstance(obj[\'explanation\'],str) or not 0<len(obj[\'explanation\'].split())<=70:raise ValueError(\'explanation\')\n    if not isinstance(obj[\'patch\'],list) or len(obj[\'patch\'])>4:raise ValueError(\'edit count\')\n    # Check full semantics separately on a disposable copy; empty patches count as abstention.\n    return obj\n\n\nclass Session:\n    def __init__(self,client,manifest):\n        self.client=client;self.manifest=manifest;self.journal=[]\n        self.calls=0;self.in_tokens=0;self.out_tokens=0\n        self.outdir=OUTDIR\n\n    def write_log(self):atomic_json(self.outdir/\'ACTION_JOURNAL.json\',self.journal)\n\n    def create(self,case,phase,**kwargs):\n        time_window(self.manifest)\n        if self.calls>=MAX_RESPONSES or self.in_tokens>=MAX_INPUT_TOKENS or self.out_tokens>=MAX_OUTPUT_TOKENS:\n            raise RuntimeError(\'global API budget exceeded\')\n        self.calls+=1\n        try:\n            r=self.client.responses.create(model=MODEL,reasoning=REASONING,max_output_tokens=OUTPUT_CAP,\n                parallel_tool_calls=False,**kwargs)\n        except Exception as e:\n            self.journal.append({\'kind\':\'api_exception\',\'case\':case,\'phase\':phase,\'exception_type\':reject_secret(e)})\n            self.write_log()\n            raise RuntimeError(\'API exception \'+reject_secret(e)) from None\n        u=getattr(r,\'usage\',None)\n        usage=u.model_dump() if hasattr(u,\'model_dump\') else {}\n        self.in_tokens+=(usage or {}).get(\'input_tokens\',0)\n        self.out_tokens+=(usage or {}).get(\'output_tokens\',0)\n        status=getattr(r,\'status\',None)\n        self.journal.append({\'kind\':\'api_response\',\'case\':case,\'phase\':phase,\'id\':getattr(r,\'id\',None),\n            \'model\':getattr(r,\'model\',None),\'status\':status,\n            \'incomplete_details\':str(getattr(r,\'incomplete_details\',None)),\n            \'usage\':usage,\'output_types\':[(getattr(o,\'type\',None),getattr(o,\'name\',None)) for o in (getattr(r,\'output\',[]) or [])]})\n        self.write_log()\n        if status!=\'completed\':raise RuntimeError(\'response not completed; \'+str(getattr(r,\'incomplete_details\',None)))\n        return r\n\n    def dispatch(self,case,stage,response,expected=None):\n        calls=[o for o in response.output if getattr(o,\'type\',None)==\'function_call\']\n        if len(calls)!=1:raise RuntimeError(\'expected exactly one native call; received \'+str(len(calls)))\n        c=calls[0]\n        if expected is not None and c.name!=expected:raise RuntimeError(\'incorrect tool choice\')\n        try:\n            args=json.loads(c.arguments)\n            if not isinstance(args,dict):raise ValueError(\'non-object args\')\n            p=make_case(case)\n            if stage==\'paired\':\n                if args!={}:raise ValueError(\'paired tool accepts no args\')\n                value={\'inspect_output_weights\':inspect_output_weights,\'probe_feature1\':probe_feature1}[c.name](p)\n            else:\n                if c.name==\'inspect_weights\':\n                    if set(args)!={\'tensor\'}:raise ValueError(\'bad tensor props\')\n                    value=inspect_tensor(p,args[\'tensor\'])\n                elif c.name==\'probe_examples\':\n                    if set(args)!={\'pack\'}:raise ValueError(\'bad probe props\')\n                    value=probe_pack(p,args[\'pack\'])\n                else:raise ValueError(\'unknown tool\')\n            valid=True\n        except Exception as e:\n            value={\'error\':type(e).__name__};valid=False\n        self.journal.append({\'kind\':\'diagnostic\',\'case\':case,\'stage\':stage,\'function\':c.name,\n            \'arguments\':c.arguments,\'valid\':valid,\'return\':value})\n        self.write_log()\n        return {\'type\':\'function_call_output\',\'call_id\':c.call_id,\'output\':json.dumps(value,allow_nan=False)},valid,c.name\n\n    def pair(self,case):\n        state={\'case\':case,\'valid_calls\':0,\'kinds\':[],\'commit\':None,\'errors\':[]}\n        prior=None;inp=\'First observe the output-layer weights; then observe the predefined feature1 probe. You will commit after those observations.\'\n        for tool in [\'inspect_output_weights\',\'probe_feature1\']:\n            args={\'instructions\':PAIR_BASE,\'input\':inp,\'tools\':PAIR_TOOLS,\n                  \'tool_choice\':{\'type\':\'function\',\'name\':tool}}\n            if prior:args[\'previous_response_id\']=prior\n            resp=self.create(case,\'paired_\'+tool,**args)\n            output,valid,name=self.dispatch(case,\'paired\',resp,expected=tool)\n            prior=resp.id;inp=[output]\n            if not valid:raise RuntimeError(\'invalid paired diagnostic\')\n            state[\'valid_calls\']+=1;state[\'kinds\'].append(name)\n        r=self.create(case,\'paired_commit\',instructions=PAIR_BASE+\' \'+PAIR_COMMIT,\n                      input=inp,previous_response_id=prior,tools=PAIR_TOOLS,tool_choice=\'none\')\n        raw=getattr(r,\'output_text\',\'\') or \'\'\n        self.journal.append({\'kind\':\'model_commitment\',\'case\':case,\'stage\':\'paired\',\'raw_text\':raw[:6000]})\n        self.write_log()\n        try:\n            commit=validate_pair_commit(json.loads(raw))\n            state[\'commit\']=commit\n            truth=intervention_effect(make_case(case))\n            state[\'actual_delta_mse\']=truth\n            state[\'prediction_error\']=commit[\'predicted_delta_mse\']-truth\n            state[\'sign_correct\']=bool(np.sign(commit[\'predicted_delta_mse\'])==np.sign(truth))\n        except Exception as e:state[\'errors\'].append(\'invalid commit: \'+type(e).__name__)\n        return state\n\n    def discovery(self,case=\'D_unknown\'):\n        state={\'case\':case,\'valid_calls\':0,\'kinds\':[],\'commit\':None,\'errors\':[],\'patch_evaluation\':None}\n        prior=None; inp=\'Investigate this unknown lesion with at most four diagnostic observations.\'\n        for turn in range(DIAG_STEPS_DISCOVERY):\n            args={\'instructions\':DISCOVERY_BASE,\'input\':inp,\'tools\':DISCOVERY_TOOLS,\'tool_choice\':\'required\'}\n            if prior:args[\'previous_response_id\']=prior\n            resp=self.create(case,\'discovery_\'+str(turn+1),**args)\n            output,valid,name=self.dispatch(case,\'discovery\',resp)\n            prior=resp.id;inp=[output]\n            if valid:\n                state[\'valid_calls\']+=1\n                if name not in state[\'kinds\']:state[\'kinds\'].append(name)\n            else:state[\'errors\'].append(\'invalid \'+name)\n        final=self.create(case,\'discovery_commit\',instructions=DISCOVERY_BASE+\' \'+DISCOVERY_COMMIT,\n                          input=inp,previous_response_id=prior,tools=DISCOVERY_TOOLS,tool_choice=\'none\')\n        raw=getattr(final,\'output_text\',\'\') or \'\'\n        self.journal.append({\'kind\':\'model_commitment\',\'case\':case,\'stage\':\'discovery\',\'raw_text\':raw[:6000]})\n        self.write_log()\n        try:\n            commit=validate_discovery_commit(json.loads(raw))\n            state[\'commit\']=commit;state[\'abstained\']=not bool(commit[\'patch\'])\n            state[\'fault_region_correct\']=bool(commit[\'fault_region\']==\'W1[0]\')\n            if commit[\'patch\']:\n                p=make_case(case)\n                score=score_edits(p,commit[\'patch\'])\n                baselines=qualification()[\'cases\'][case]\n                improve=score[\'before_mse\']-score[\'after_mse\']\n                state[\'patch_evaluation\']={**score,\n                  \'beats_no_edit\':bool(improve>0),\n                  \'beats_simple_heuristic_by_0p05\':bool(score[\'after_mse\']+0.05<=baselines[\'heuristic_mse\']),\n                  \'recovery_fraction\':float(improve/score[\'before_mse\']),\n                  \'G4_pass\':bool(improve/score[\'before_mse\']>=0.50 and score[\'after_mse\']+0.05<=baselines[\'heuristic_mse\'])}\n        except Exception as e:state[\'errors\'].append(\'invalid discovery commitment or patch: \'+type(e).__name__)\n        return state\n\n\ndef evaluate(summary,journal):\n    cases=summary[\'cases\']\n    pair_complete=all(cases.get(c,{}).get(\'commit\') is not None for c in (\'C_positive\',\'C_negative\'))\n    paired_obs=all(cases.get(c,{}).get(\'valid_calls\')==2 and cases.get(c,{}).get(\'kinds\')==[\'inspect_output_weights\',\'probe_feature1\'] for c in (\'C_positive\',\'C_negative\'))\n    def exact_calls(case):return [(j[\'function\'],j[\'arguments\']) for j in journal if j.get(\'kind\')==\'diagnostic\' and j.get(\'case\')==case and j.get(\'stage\')==\'paired\' and j.get(\'valid\')]\n    matched=exact_calls(\'C_positive\')==exact_calls(\'C_negative\') and len(exact_calls(\'C_positive\'))==2\n    sign=pair_complete and all(cases[c].get(\'sign_correct\') for c in (\'C_positive\',\'C_negative\')) and cases[\'C_positive\'][\'commit\'][\'predicted_delta_mse\']<0<cases[\'C_negative\'][\'commit\'][\'predicted_delta_mse\']\n    disc=cases.get(\'D_unknown\',{})\n    summary[\'matched_pair_observation_contract\']=bool(matched)\n    summary[\'G1\']=\'PASS\' if paired_obs else \'FAIL\'\n    summary[\'G2\']=\'PASS\' if (paired_obs and matched and sign) else (\'INCONCLUSIVE\' if not matched else \'FAIL\')\n    summary[\'G3\']=\'PASS\' if pair_complete else \'FAIL\'\n    summary[\'G4\']=(\'NOT RUN\' if not disc.get(\'patch_evaluation\') else \'PASS\' if disc[\'patch_evaluation\'][\'G4_pass\'] else \'FAIL\')\n    summary[\'discovery_tool_diversity\']=\'PASS\' if len(disc.get(\'kinds\',[]))>=2 else \'FAIL\' if disc else \'NOT RUN\'\n    summary[\'mechanistic_understanding\']=\'NOT ESTABLISHED\'\n    summary[\'instrument_ready\']=bool(summary[\'G1\']==\'PASS\' and summary[\'G2\']==\'PASS\' and summary[\'G3\']==\'PASS\')\n    summary[\'status\']=\'COMPLETE\' if not any(v.get(\'interruption\') for v in cases.values()) and len(cases)==3 else \'PARTIAL / INTERRUPTED\'\n    return summary\n\n\ndef run_live():\n    manifest=load_json(OUTDIR/\'PRE_LIVE_FREEZE.json\')\n    if manifest.get(\'experiment\')!=\'EX0005Aa\':raise RuntimeError(\'wrong manifest\')\n    if manifest.get(\'source_sha256\')!=code_hashes() or manifest.get(\'public_fixture_fingerprint\')!=qualification()[\'fixture_sha256\']:\n        raise RuntimeError(\'source or fixtures changed since freeze; no model contacted\')\n    if manifest.get(\'freeze_sha256\')!=fingerprint({k:v for k,v in manifest.items() if k!=\'freeze_sha256\'}):raise RuntimeError(\'manifest modified since freeze\')\n    time_window(manifest)\n    marker=OUTDIR/\'LIVE_ATTEMPT_STARTED.txt\'\n    if marker.exists() or (OUTDIR/\'RESULTS.json\').exists() or (OUTDIR/\'ACTION_JOURNAL.json\').exists():\n        raise RuntimeError(\'live attempt exists: no automatic reruns\')\n    marker.write_text(local_date_time().isoformat()+\'\\n\')\n    summary={\'experiment\':\'EX0005Aa\',\'model_requested\':MODEL,\'freeze_sha256\':manifest[\'freeze_sha256\'],\n       \'G0_local\':\'PASS\',\'cases\':{},\'G1\':\'NOT RUN\',\'G2\':\'NOT RUN\',\'G3\':\'NOT RUN\',\'G4\':\'NOT RUN\',\'instrument_ready\':False}\n    session=None\n    try:\n        from openai import OpenAI\n        client=OpenAI(api_key=api_key(),max_retries=0,timeout=45.0)\n        session=Session(client,manifest)\n        try:\n            r=client.models.retrieve(MODEL)\n            summary[\'model_access_check\']=getattr(r,\'id\',None)\n        except Exception as e:\n            raise RuntimeError(\'model access failed: \'+type(e).__name__) from None\n        for case in (\'C_positive\',\'C_negative\',\'D_unknown\'):\n            try:\n                if case.startswith(\'C_\'):out=session.pair(case)\n                else:out=session.discovery(case)\n                summary[\'cases\'][case]=out\n            except Exception as e:\n                # Reconstruct already accepted observations from the immutable journal;\n                # an interrupted case is partial, never silently called NOT RUN.\n                accepted=[j for j in session.journal if j.get(\'case\')==case and j.get(\'kind\')==\'diagnostic\' and j.get(\'valid\')]\n                summary[\'cases\'][case]={\'case\':case,\'interruption\':{\'type\':type(e).__name__,\'reason\':str(e)[:300]},\n                                       \'commit\':None,\'valid_calls\':len(accepted),\n                                       \'kinds\':list(dict.fromkeys(j[\'function\'] for j in accepted))}\n            finally:\n                summary[\'api_responses_logged\']=sum(j[\'kind\']==\'api_response\' for j in session.journal)\n                atomic_json(OUTDIR/\'RESULTS.json\',summary)\n        summary=evaluate(summary,session.journal)\n    except Exception as e:\n        summary[\'interruption\']={\'type\':type(e).__name__,\'reason\':str(e)[:300]}\n    finally:\n        if session:\n            summary[\'api_responses_logged\']=sum(j[\'kind\']==\'api_response\' for j in session.journal)\n            summary[\'usage_input_tokens\']=session.in_tokens;summary[\'usage_output_tokens\']=session.out_tokens\n            session.write_log()\n        atomic_json(OUTDIR/\'RESULTS.json\',summary)\n    return summary\n\n\ndef main():\n    ap=argparse.ArgumentParser()\n    g=ap.add_mutually_exclusive_group(required=True)\n    g.add_argument(\'--qualify-local\',action=\'store_true\')\n    g.add_argument(\'--freeze\',action=\'store_true\')\n    g.add_argument(\'--live\',action=\'store_true\')\n    ap.add_argument(\'--human-fields\',default=\'human_fields.json\')\n    ap.add_argument(\'--session-date\')\n    ap.add_argument(\'--start\')\n    ap.add_argument(\'--end\')\n    args=ap.parse_args()\n    if args.qualify_local:\n        print(json.dumps(qualification(),indent=2));return\n    if args.freeze:\n        if not (args.session_date and args.start and args.end):ap.error(\'freeze requires session date, start and end local\')\n        cfg=human_fields(args.human_fields)\n        print(json.dumps(freeze(cfg,args.session_date,args.start,args.end),indent=2));return\n    print(json.dumps(run_live(),indent=2))\n\nif __name__==\'__main__\':main()\n', encoding='utf-8')
print('Wrote', 'run_luna_aa.py', Path('run_luna_aa.py').stat().st_size, 'bytes')

In [ ]:
from pathlib import Path
Path('test_offline_aa.py').write_text('"""Offline tests, including mock Response API protocol checks. No API token read."""\nimport json, pathlib, tempfile, types, unittest\nfrom unittest.mock import patch\nimport numpy as np\nimport ex0005aa_public as p\nimport run_luna_aa as r\n\nclass FakeUsage:\n    def model_dump(self):return {\'input_tokens\':50,\'output_tokens\':30}\n\nclass FakeResp:\n    def __init__(self,i,name=None,arguments=\'{}\',text=None,status=\'completed\'):\n        self.id=f\'fake_{i}\';self.model=\'FAKE_NOT_LUNA\';self.status=status\n        self.incomplete_details=\'max_messages\' if status!=\'completed\' else None\n        self.usage=FakeUsage();self.output_text=text or \'\'\n        self.output=([types.SimpleNamespace(type=\'function_call\',name=name,arguments=arguments,call_id=f\'call_{i}\')] if name else [])\n\nclass FakeResponses:\n    def __init__(self,queue):self.queue=list(queue);self.request_log=[]\n    def create(self,**kwargs):\n        self.request_log.append(kwargs)\n        return self.queue.pop(0)\n\nclass Tests(unittest.TestCase):\n    def test_fixture_and_oracles(self):\n        q=p.qualification()\n        self.assertEqual(q[\'status\'],\'G0_LOCAL_PASS_ONLY\')\n        self.assertLess(q[\'cases\'][\'C_positive\'][\'counterfactual_delta_mse\'],0)\n        self.assertGreater(q[\'cases\'][\'C_negative\'][\'counterfactual_delta_mse\'],0)\n        self.assertGreater(q[\'cases\'][\'D_unknown\'][\'heuristic_mse\'],0.5)\n        self.assertEqual(q[\'cases\'][\'D_unknown\'][\'oracle_mse\'],0)\n    def test_paired_single_change(self):\n        a=p.make_case(\'C_positive\');b=p.make_case(\'C_negative\')\n        self.assertTrue(np.array_equal(a.x_eval,b.x_eval))\n        self.assertTrue(np.array_equal(a.y_eval,b.y_eval))\n        for t in (\'W1\',\'b1\',\'b2\'):\n            self.assertTrue(np.array_equal(a.w[t],b.w[t]))\n        self.assertEqual(a.w[\'W2\'][0],b.w[\'W2\'][0])\n    def test_bad_values_rejected(self):\n        patient=p.make_case(\'D_unknown\')\n        for fn in [lambda:p.inspect_tensor(patient,\'bad\'),lambda:p.probe_pack(patient,\'oops\'),\n                   lambda:p.apply_edit(patient,[{\'tensor\':\'W1\',\'index\':0,\'delta\':500}]),\n                   lambda:p.apply_edit(patient,[{\'tensor\':\'W1\',\'index\':0,\'delta\':0.2},{\'tensor\':\'W1\',\'index\':0,\'delta\':0.3}])]:\n            with self.assertRaises(ValueError):fn()\n    def test_numeric_commit_reject_wrong(self):\n        ok={\'predicted_delta_mse\':-.2,\'uncertainty\':\'high\',\'explanation\':\'Tentatively correct sign\'}\n        self.assertEqual(r.validate_pair_commit(ok),ok)\n        for bad in [{**ok,\'predicted_delta_mse\':float(\'nan\')},{**ok,\'explanation\':\'\'},{**ok,\'unrecognized\':1}]:\n            with self.assertRaises(ValueError):r.validate_pair_commit(bad)\n    def test_discovery_commit_validator(self):\n        ok={\'fault_region\':\'W1[0]\',\'uncertainty\':\'high\',\'patch\':[],\'explanation\':\'Insufficient evidence\'}\n        self.assertEqual(r.validate_discovery_commit(ok),ok)\n        with self.assertRaises(ValueError):r.validate_discovery_commit({**ok,\'fault_region\':\'W1[99]\'})\n    def test_human_origin_lock(self):\n        with tempfile.TemporaryDirectory() as tmp:\n            f=pathlib.Path(tmp)/\'fields.json\'\n            d={k:\'This prediction belongs to the earlier completed session.\' for k in r.human_fields.__code__.co_consts if False}\n            d={k:\'This is an existing human hypothesis from our earlier session.\' for k in (\'human_predicted_failure\',\'human_predicted_contrast\',\'human_false_positive_test\',\'human_unsupported_mechanism\')}\n            d.update(assistance_disclosure=\'C\',origin=\'carried_from_EX0005A_pre_live\',approved_as_prior_for_aa=False)\n            f.write_text(json.dumps(d))\n            with self.assertRaises(ValueError):r.human_fields(f)\n            d[\'approved_as_prior_for_aa\']=True;f.write_text(json.dumps(d))\n            self.assertEqual(r.human_fields(f)[\'assistance_disclosure\'],\'C\')\n    def test_time_window_reject_closed_session(self):\n        with self.assertRaises(RuntimeError):r.time_window({\'session_date\':\'2026-11-15\',\'session_start\':\'18:00\',\'session_end\':\'21:00\'})\n    def test_pair_real_call_contract_mock(self):\n        responses=FakeResponses([\n            FakeResp(1,\'inspect_output_weights\'),FakeResp(2,\'probe_feature1\'),\n            FakeResp(3,text=json.dumps({\'predicted_delta_mse\':-0.2,\'uncertainty\':\'high\',\'explanation\':\'Observed feature1 overprediction.\'}))\n        ])\n        with tempfile.TemporaryDirectory() as tmp,patch.object(r,\'time_window\',lambda manifest:None):\n            sess=r.Session(types.SimpleNamespace(responses=responses),{\'session_date\':\'2026-11-19\'})\n            sess.outdir=pathlib.Path(tmp)\n            state=sess.pair(\'C_positive\')\n            self.assertEqual(state[\'valid_calls\'],2)\n            self.assertEqual(state[\'kinds\'],[\'inspect_output_weights\',\'probe_feature1\'])\n            self.assertTrue(state[\'sign_correct\'])\n            self.assertEqual(len(responses.request_log),3)\n            self.assertTrue(all(req[\'parallel_tool_calls\'] is False for req in responses.request_log))\n            self.assertTrue((pathlib.Path(tmp)/\'ACTION_JOURNAL.json\').exists())\n    def test_discovery_can_beat_baseline_mock(self):\n        picks=[(\'inspect_weights\',\'{"tensor":"W1"}\'),(\'probe_examples\',\'{"pack":"feature0"}\'),\n               (\'inspect_weights\',\'{"tensor":"W2"}\'),(\'probe_examples\',\'{"pack":"mixed"}\')]\n        queue=[FakeResp(i+1,name,arguments) for i,(name,arguments) in enumerate(picks)]\n        queue.append(FakeResp(5,text=json.dumps({\'fault_region\':\'W1[0]\',\'uncertainty\':\'high\',\n                   \'patch\':[{\'tensor\':\'W1\',\'index\':0,\'delta\':1.3}],\n                   \'explanation\':\'The first hidden feature is suppressed; the edit would restore response.\'})))\n        responses=FakeResponses(queue)\n        with tempfile.TemporaryDirectory() as tmp,patch.object(r,\'time_window\',lambda manifest:None):\n            sess=r.Session(types.SimpleNamespace(responses=responses),{\'session_date\':\'2026-11-19\'})\n            sess.outdir=pathlib.Path(tmp)\n            state=sess.discovery()\n            self.assertEqual(state[\'valid_calls\'],4)\n            self.assertTrue(state[\'patch_evaluation\'][\'G4_pass\'])\n            self.assertTrue(all(req[\'parallel_tool_calls\'] is False for req in responses.request_log))\n    def test_incomplete_retained_not_passed(self):\n        responses=FakeResponses([FakeResp(1,\'inspect_output_weights\',status=\'incomplete\')])\n        with tempfile.TemporaryDirectory() as tmp,patch.object(r,\'time_window\',lambda manifest:None):\n            sess=r.Session(types.SimpleNamespace(responses=responses),{\'session_date\':\'2026-11-19\'})\n            sess.outdir=pathlib.Path(tmp)\n            with self.assertRaises(RuntimeError):sess.pair(\'C_positive\')\n            j=json.loads((pathlib.Path(tmp)/\'ACTION_JOURNAL.json\').read_text())\n            self.assertEqual(j[0][\'status\'],\'incomplete\')\n            self.assertEqual(j[0][\'incomplete_details\'],\'max_messages\')\n    def test_summary_does_not_infer_success_from_incomplete(self):\n        results={\'cases\':{\'C_positive\':{\'valid_calls\':2,\'kinds\':[\'inspect_output_weights\',\'probe_feature1\'],\n                   \'commit\':{\'predicted_delta_mse\':-0.1},\'sign_correct\':True},\n                   \'C_negative\':{\'valid_calls\':0,\'kinds\':[],\'commit\':None,\'interruption\':{\'reason\':\'incomplete\'}}}}\n        out=r.evaluate(results,[])\n        self.assertEqual(out[\'G1\'],\'FAIL\')\n        self.assertEqual(out[\'G3\'],\'FAIL\')\n        self.assertFalse(out[\'instrument_ready\'])\n\nif __name__==\'__main__\':unittest.main(verbosity=2)\n', encoding='utf-8')
print('Wrote', 'test_offline_aa.py', Path('test_offline_aa.py').stat().st_size, 'bytes')

## Cell 3 · Local G0 and independent negative controls
Expected: new fixtures have measurable lesions, oracle repairs exactly, naive baselines are scored, and 11 offline tests pass. **Mocks are not Luna results.**

In [ ]:
import subprocess, sys
for cmd in [[sys.executable,'ex0005aa_public.py'],[sys.executable,'-m','unittest','-v','test_offline_aa']]:
    proc=subprocess.run(cmd, capture_output=True, text=True)
    print(proc.stdout[-6500:]);print(proc.stderr[-4500:]);print('exit status',proc.returncode)
    if proc.returncode:raise RuntimeError('Local G0 failed. DO NOT freeze or run Luna.')

## Cell 4 · Review inherited human predictions
These statements were written **before EX0005A**, but are carried forward as **prior hypotheses** into EX0005Aa. They are not fresh independent predictions. You may add a separate new prediction in your research notes. Set `approved_as_prior_for_aa=True` only after reading and approving the reuse. Assistance was **C, collaborative**.

In [ ]:
import json
from pathlib import Path
prior_fields={'human_predicted_failure': 'I predict Luna may improve network performance through a heuristic or numerical shortcut, but fail to correctly explain WHY the repair works. This resembles our earlier experiments, where functional improvement did not establish mechanistic understanding.', 'human_predicted_contrast': 'I seriously doubt Luna will correctly recognize the causal difference between P1_plus and P1_minus. I predict it may fail to anticipate that decreasing the same weight improves P1_plus but worsens P1_minus.', 'human_false_positive_test': 'The evaluator could falsely reward Luna if the prompt, examples, or simple heuristics supply the answer. Deliberately submit an incorrect directional prediction and verify that the checker rejects it. Ensure the model cannot see the oracle answer.', 'human_unsupported_mechanism': 'Luna might confidently identify a supposed underlying cause simply because a patch improves performance. But a successful patch does not prove the model understands the mechanism. Its explanation could be plausible but unsupported by causal evidence.', 'assistance_disclosure': 'C', 'origin': 'carried_from_EX0005A_pre_live', 'approved_as_prior_for_aa': False}
prior_fields['approved_as_prior_for_aa'] = False  # review, then explicitly change to True
Path('human_fields.json').write_text(json.dumps(prior_fields,indent=2))
print('Prior hypotheses carried, approved:', prior_fields['approved_as_prior_for_aa'])

## Cell 5 · Freeze a **future separate session** (no billable calls)
Choose date after November 15, 2026, local start/end HH:MM, no longer than 4 hours. This notebook does not automatically select your next available time. Once frozen, **do not modify source files, metrics, budgets or predictions**. Save the manifest. If the freeze fails, inspect it before proceeding.

In [ ]:
import sys, subprocess
SESSION_DATE = "YYYY-MM-DD" # e.g. choose an actually available future session, not November 15
SESSION_START = "09:00"
SESSION_END = "11:00"
FREEZE_NOW = False  # explicitly change True once fields and local tests are checked

if FREEZE_NOW:
    command=[sys.executable,'run_luna_aa.py','--freeze',
             '--session-date',SESSION_DATE,'--start',SESSION_START,'--end',SESSION_END]
    proc=subprocess.run(command,capture_output=True,text=True)
    print(proc.stdout[-5000:])
    print(proc.stderr[-1000:])
    print('Freeze exit status:',proc.returncode)
else:
    print('Freeze intentionally OFF. No Luna calls made.')

## Cell 6 · Real Luna API investigation (BILLABLE OPT-IN, NEW SESSION ONLY)
First confirm your frozen manifest, its code/fixture hashes and local wall clock. Add `OPENAI_API_KEY` to **Colab Secrets** and grant notebook access. This cell bridges it into the subprocess environment, avoiding `NoneType.kernel`, but never prints or saves it. The runner refuses execution outside the frozen window or after a previous attempt. Real generations may be billed even if incomplete.

**Note:** `RUN_BILLABLE_API` defaults False. Enabling it is a new, separate experiment, not a continuation of November 15.

In [ ]:
import os, sys, subprocess, json
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo

RUN_BILLABLE_API=False  # deliberately OFF; change True only within the separate frozen session
manifest_path=Path('ex0005aa_receipts/PRE_LIVE_FREEZE.json')
print('Local Denver time:',datetime.now(ZoneInfo('America/Denver')).isoformat())
print('Freeze exists:',manifest_path.exists())
if manifest_path.exists():
    manifest=json.loads(manifest_path.read_text())
    print('Freeze:',manifest['freeze_sha256'])
    print('Session:',manifest['session_date'],manifest['session_start'],manifest['session_end'])

if RUN_BILLABLE_API:
    if not manifest_path.exists():raise RuntimeError('Must freeze before billable run')
    from google.colab import userdata
    secret=userdata.get('OPENAI_API_KEY')
    if not secret:raise RuntimeError('Enable secret access; do not paste token in notebook')
    env=os.environ.copy();env['OPENAI_API_KEY']=secret
    try:
        proc=subprocess.run([sys.executable,'run_luna_aa.py','--live'],env=env,
                            capture_output=True,text=True)
        print('Runner exit status:',proc.returncode)
        print('RESULT SUMMARY:',proc.stdout.replace(secret,'[REDACTED]')[-6500:])
        if proc.returncode:print('ERROR TYPE/DETAILS:',proc.stderr.replace(secret,'[REDACTED]')[-1000:])
    finally:
        env.pop('OPENAI_API_KEY',None)
        del secret, env
else:
    print('Live Luna deliberately NOT RUN.')

## Cell 7 · Read-only evidence inspection
G0 controls, model-use gates, error denominators, predictions and patch scores are **separate**. Do not score an interruption as model inability. One public pair and one public discovery case are descriptive only.

In [ ]:
from pathlib import Path
import json
folder=Path('ex0005aa_receipts')
for name in ['PRE_LIVE_FREEZE.json','RESULTS.json','ACTION_JOURNAL.json']:
    path=folder/name
    if not path.exists():print(name,': MISSING');continue
    data=json.loads(path.read_text())
    print('
',name)
    if name=='ACTION_JOURNAL.json':
        from collections import Counter
        print('Total records:',len(data),'kinds:',dict(Counter(r.get('kind') for r in data)))
        for case in ['C_positive','C_negative','D_unknown']:
            subset=[r for r in data if r.get('case')==case and r.get('kind')=='diagnostic']
            print(case,'valid:',sum(r.get('valid') is True for r in subset),'attempted:',len(subset))
    elif name=='RESULTS.json':
        print(json.dumps(data,indent=2)[:9000])
    else:
        print('Freeze SHA:',data.get('freeze_sha256'),'source:',data.get('source_sha256'))
print('READ-ONLY. No new API calls.')

## Cell 8 · Export receipts without overwriting EX0005A
Run after the new session. Save the ZIP alongside the original evidence. An interrupted experiment is a legitimate finished receipt.

In [ ]:
import shutil
from pathlib import Path
folder=Path('ex0005aa_receipts')
if folder.exists():
    archive=shutil.make_archive('EX0005Aa_receipts','zip',root_dir='.',base_dir='ex0005aa_receipts')
    print('Receipt ZIP:',archive)
else:
    print('No EX0005Aa receipt yet; no model trial ran.')